# Speech Command Classifier: SraVaani ASR → Laya Decision Model

This notebook builds a pipeline that:

1. Takes an audio clip (Hindi, English, or mixed) as input.
2. Transcribes it with **SraVaani-1.0** (ARTPARK-IISc), a 65-language Indic ASR model.
3. Feeds the transcript to **Laya** (`convaiinnovations/laya`, multilingual checkpoint), a typed-decision model, with a `choice` question over six labels: `left`, `right`, `forward`, `backward`, `stop`, `other`.
4. Applies a confidence threshold and a keyword safety net for the `stop` command.
5. Includes a small evaluation harness so you can measure accuracy on your own labeled clips before trusting this in production.

**Before you run this:**
- Visit https://huggingface.co/ARTPARK-IISc/SraVaani-1.0 and accept the gated-access conditions with your Hugging Face account.
- Generate a Hugging Face access token (read scope is enough): https://huggingface.co/settings/tokens
- Have a GPU available if you want reasonable latency; both models together are large (~1.6 GB) and will run on CPU, just slower.

**Memory note:** SraVaani (~900 MB, FP16) and Laya-multilingual (~322M params, mmBERT-base) are both loaded into memory at once and kept resident for the life of the process. Budget accordingly on constrained devices.


## 1. Environment setup

Install the packages this notebook needs. Restart the kernel afterward if `transformers` or `torch` were freshly installed.

In [4]:
%pip uninstall -y torch torchvision
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cpu
# Core ML stack
%pip install -q --upgrade transformers accelerate soundfile librosa

# Laya's local runtime (typed decision model)

%pip install -q --upgrade laya

# Hugging Face Hub client, for the gated SraVaani model
%pip install -q "huggingface_hub<2.0,>=1.23.0"


Found existing installation: torch 2.14.0+cpu
Uninstalling torch-2.14.0+cpu:
  Successfully uninstalled torch-2.14.0+cpu
Found existing installation: torchvision 0.29.0+cpu
Uninstalling torchvision-0.29.0+cpu:
  Successfully uninstalled torchvision-0.29.0+cpu


### 1.1 Confirm versions

Sanity-check that everything imported and print versions, so a stale cached package doesn't silently break the pipeline later.

In [5]:
import torch, transformers, sys

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA avail. :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))

try:
    import laya
    print("laya module loaded OK")
except ImportError as e:
    print("laya import failed:", e)


Python      : 3.13.15
PyTorch     : 2.14.0+cpu
Transformers: 5.17.0
CUDA avail. : False
laya module loaded OK


## 2. Authenticate with Hugging Face

SraVaani-1.0 is a **gated** model. You must have accepted its access conditions on the model page before this will work. Paste your token below, or set the `HF_TOKEN` environment variable beforehand and skip the prompt.


In [6]:
import os
from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN") or input("Paste your Hugging Face access token: ").strip()
login(token=HF_TOKEN)
print("Logged in to Hugging Face Hub.")


Logged in to Hugging Face Hub.


## 3. Load the ASR model (SraVaani-1.0)

~430M parameters, FP16, ~900 MB on disk. First run downloads the weights, which can take a while depending on your connection.


In [7]:
from transformers import AutoModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ASR_REPO = "ARTPARK-IISc/SraVaani-1.0"

print(f"Loading {ASR_REPO} on {DEVICE} ...")
asr_model = AutoModel.from_pretrained(
    ASR_REPO,
    trust_remote_code=True,
    token=HF_TOKEN,
).to(DEVICE).eval()

print("ASR model loaded.")


Loading ARTPARK-IISc/SraVaani-1.0 on cpu ...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

ASR model loaded.


### 3.1 Quick ASR smoke test

Point this at any short local WAV/FLAC/MP3 file (16 kHz mono is safest) to confirm transcription works before wiring up the full pipeline.

In [ ]:
TEST_AUDIO_PATH = "path/to/your/test_clip.wav"  # <-- change this

def transcribe(path: str) -> str:
    """Run SraVaani ASR on a single audio file and return the transcript text."""
    hyps = asr_model.transcribe(path, return_hypotheses=True)
    return hyps[0].text

if os.path.exists(TEST_AUDIO_PATH):
    text = transcribe(TEST_AUDIO_PATH)
    print("Transcript:", text)
else:
    print(f"No file found at {TEST_AUDIO_PATH} -- update the path above and re-run this cell.")


## 4. Load the decision model (Laya)

We use the **multilingual** checkpoint (mmBERT-base) since traffic here is Hindi, English, or code-mixed. Laya's `Router` can auto-select a checkpoint, but for short Hinglish commands the automatic language/script routing can be ambiguous, so we pin the multilingual model explicitly.


In [8]:
from laya import Router

router = Router(default="multilingual")
print("Laya router ready (default checkpoint: multilingual).")


Laya router ready (default checkpoint: multilingual).


### 4.1 Define the command schema

One `choice` question with six labels. The `criteria` strings give the model concrete cues in both languages — extend these with more transliterations/synonyms you expect from real users.


In [9]:
COMMAND_QUESTIONS = {
    "command": {
        "type": "choice",
        "instructions": "Which movement command does the speaker give?",
        "criteria": {
            "left":     "turn or move left (बाएं, baayen, left, ulta)",
            "right":    "turn or move right (दाएं, daayen, right)",
            "forward":  "go ahead or straight (आगे, aage, forward, seedha, straight)",
            "backward": "go back or reverse (पीछे, peeche, back, backward, reverse)",
            "stop":     "halt or stop moving (रुको, ruko, ruk jao, stop, halt)",
            "other":    "anything that is not one of the above five commands",
        },
    }
}

print("Command schema defined:", list(COMMAND_QUESTIONS["command"]["criteria"].keys()))


Command schema defined: ['left', 'right', 'forward', 'backward', 'stop', 'other']


## 5. Full pipeline: audio → text → label

Combines ASR and Laya into one function, with:
- a **confidence threshold** (`CONF_MIN`) that forces low-confidence predictions to `other`,
- a **keyword safety net** for `stop`, since a missed stop command is the costliest failure mode in most command-driven systems.

Tune `CONF_MIN` and the keyword lists against your own evaluation set in Section 6 — don't ship the defaults untested.


In [10]:
CONF_MIN = 0.6

STOP_KEYWORDS = ["ruko", "ruk jao", "रुको", "रुक जाओ", "stop", "halt"]

def _extract_choice_and_confidence(result: dict):
    """
    Pull the chosen label and its probability out of a Laya `predict` result.

    NOTE: field names below follow the upstream Router.predict shape as
    documented at the time of writing. If your installed `laya` version
    differs, print(result) once and adjust the keys accordingly.
    """
    ans = result["answers"]["command"]
    label = ans["choice"]
    probs = ans.get("probabilities") or ans.get("distribution") or {}
    confidence = probs.get(label, None)
    return label, confidence, probs

def keyword_stop_override(text: str) -> bool:
    lowered = text.lower()
    return any(kw in lowered for kw in STOP_KEYWORDS)

def pipeline(audio_path: str, verbose: bool = False):
    """
    Full audio -> transcript -> command label pipeline.

    Returns a dict with: transcript, label, confidence, raw_result.
    """
    transcript = transcribe(audio_path)

    result = router.predict({"utterance": transcript}, COMMAND_QUESTIONS)
    label, confidence, probs = _extract_choice_and_confidence(result)

    if confidence is not None and confidence < CONF_MIN:
        if verbose:
            print(f"Confidence {confidence:.2f} below threshold {CONF_MIN}, forcing 'other'")
        label = "other"

    if keyword_stop_override(transcript) and label != "stop":
        if verbose:
            print("Keyword safety net triggered: overriding to 'stop'")
        label = "stop"

    if verbose:
        print("Transcript :", transcript)
        print("Label      :", label)
        print("Confidence :", confidence)
        print("Full dist. :", probs)

    return {
        "transcript": transcript,
        "label": label,
        "confidence": confidence,
        "raw_result": result,
    }


### 5.1 Try it on a single file

In [ ]:
if os.path.exists(TEST_AUDIO_PATH):
    out = pipeline(TEST_AUDIO_PATH, verbose=True)
else:
    print(f"No file found at {TEST_AUDIO_PATH} -- update the path in Section 3.1 and re-run.")


## 6. Batch inference over a folder of clips

Point this at a directory of audio files (one command per file) to run the pipeline over all of them at once. Useful for a quick spot-check before building a formal evaluation set.


In [ ]:
from pathlib import Path

AUDIO_DIR = Path("path/to/audio_folder")  # <-- change this
AUDIO_EXTS = {".wav", ".flac", ".mp3", ".m4a", ".ogg"}

def run_batch(folder: Path):
    results = []
    if not folder.exists():
        print(f"Folder not found: {folder}")
        return results
    files = sorted(p for p in folder.iterdir() if p.suffix.lower() in AUDIO_EXTS)
    print(f"Found {len(files)} audio files in {folder}")
    for p in files:
        out = pipeline(str(p))
        out["file"] = p.name
        results.append(out)
        print(f"{p.name:40s} -> {out['label']:9s} (conf={out['confidence']})  | '{out['transcript']}'")
    return results

batch_results = run_batch(AUDIO_DIR)


## 7. Evaluation harness

Before trusting this pipeline for anything safety-relevant, measure it against a labeled set. Build a CSV with two columns: `filename` and `true_label` (one of `left`, `right`, `forward`, `backward`, `stop`, `other`), covering both languages, code-mixing, and background noise you'll see in production.

This section computes overall accuracy, a confusion matrix, and per-class precision/recall — and flags `stop` misses specifically, since those are the costliest errors.


In [ ]:
import csv

LABELS = ["left", "right", "forward", "backward", "stop", "other"]
LABELED_CSV = "path/to/labels.csv"  # <-- change this: columns = filename, true_label
LABELED_AUDIO_DIR = Path("path/to/audio_folder")  # same folder as above, or a separate eval set

def load_labels(csv_path: str):
    rows = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append((row["filename"], row["true_label"].strip().lower()))
    return rows


In [ ]:
def evaluate(csv_path: str, audio_dir: Path):
    if not os.path.exists(csv_path):
        print(f"Labels CSV not found at {csv_path} -- create it first (see markdown above).")
        return None

    rows = load_labels(csv_path)
    y_true, y_pred, records = [], [], []

    for filename, true_label in rows:
        fpath = audio_dir / filename
        if not fpath.exists():
            print(f"Skipping missing file: {fpath}")
            continue
        out = pipeline(str(fpath))
        y_true.append(true_label)
        y_pred.append(out["label"])
        records.append({**out, "file": filename, "true_label": true_label})

    return y_true, y_pred, records

eval_out = evaluate(LABELED_CSV, LABELED_AUDIO_DIR)


### 7.1 Confusion matrix and per-class metrics

In [ ]:
import numpy as np
import pandas as pd

def confusion_matrix_df(y_true, y_pred, labels):
    idx = {l: i for i, l in enumerate(labels)}
    mat = np.zeros((len(labels), len(labels)), dtype=int)
    for t, p in zip(y_true, y_pred):
        if t in idx and p in idx:
            mat[idx[t], idx[p]] += 1
    return pd.DataFrame(mat, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])

def per_class_metrics(y_true, y_pred, labels):
    rows = []
    for l in labels:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == l and p == l)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != l and p == l)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == l and p != l)
        precision = tp / (tp + fp) if (tp + fp) else float("nan")
        recall = tp / (tp + fn) if (tp + fn) else float("nan")
        rows.append({"label": l, "precision": precision, "recall": recall, "support": tp + fn})
    return pd.DataFrame(rows)

if eval_out is not None:
    y_true, y_pred, records = eval_out
    accuracy = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true) if y_true else float("nan")
    print(f"Overall accuracy: {accuracy:.3f} over {len(y_true)} examples\n")

    print("Confusion matrix:")
    display(confusion_matrix_df(y_true, y_pred, LABELS))

    print("\nPer-class precision/recall:")
    display(per_class_metrics(y_true, y_pred, LABELS))

    stop_misses = [r for r in records if r["true_label"] == "stop" and r["label"] != "stop"]
    print(f"\nSTOP commands missed: {len(stop_misses)}")
    for r in stop_misses:
        print(f"  {r['file']}: transcript='{r['transcript']}' predicted='{r['label']}'")


### 7.2 Calibration check (optional but recommended)

The Laya documentation is explicit that confidence values need checking, not assuming. This cell buckets predictions by confidence and compares each bucket's average confidence to its observed accuracy — a simple reliability check. If a 0.8-confidence bucket is only 50% correct, your `CONF_MIN` threshold needs to move up.


In [ ]:
def reliability_table(records, n_bins=5):
    scored = [r for r in records if r["confidence"] is not None]
    if not scored:
        print("No confidence values available to calibrate -- check the field-name mapping in Section 5.")
        return None

    bins = np.linspace(0, 1, n_bins + 1)
    rows = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        bucket = [r for r in scored if lo <= r["confidence"] < hi or (hi == 1.0 and r["confidence"] == 1.0)]
        if not bucket:
            continue
        avg_conf = np.mean([r["confidence"] for r in bucket])
        acc = np.mean([r["label"] == r["true_label"] for r in bucket])
        rows.append({"bucket": f"[{lo:.1f}, {hi:.1f})", "n": len(bucket), "avg_confidence": avg_conf, "observed_accuracy": acc})
    return pd.DataFrame(rows)

if eval_out is not None:
    _, _, records = eval_out
    table = reliability_table(records)
    if table is not None:
        display(table)


## 8. Next steps

- **Tune `CONF_MIN`** using the reliability table above rather than the placeholder value of 0.6.
- **Expand `STOP_KEYWORDS`** with any additional transliterations or dialect words your users actually say.
- **Watch left/right confusion specifically** — short command words are the most common ASR error class, and it's the costliest one to get wrong if this drives a physical system.
- **Re-run Section 7 whenever you change the prompt, threshold, or either model version** — treat it as a regression test, not a one-time check.
- **Consider shadow-mode deployment**: log predictions alongside the existing control method before letting this pipeline take real actions.


## 9. Real-time pipeline with a wake word

Everything above runs on a pre-recorded file. This section turns it into a live loop:
microphone → wake word detector → record the command → run the existing `pipeline()`.

**Requirements:**
- A **local Jupyter kernel** (not Colab's hosted runtime) — this needs direct access to your machine's microphone.
- A working microphone recognized by `sounddevice`.

**How it works:**
1. Continuously listen in small 80&nbsp;ms audio blocks.
2. Score each block with [openWakeWord](https://github.com/dscripka/openWakeWord) (pretrained `"hey_jarvis"` model, offline, no API key).
3. When the wake word score crosses a threshold, switch to recording mode and buffer audio until the speaker goes quiet (simple energy-based silence timeout) or a max duration is hit.
4. Write that clip to a temp WAV file and run it through the same `pipeline()` from Section 5 (SraVaani → Laya → label).
5. Print the result and go back to listening.

Stop the loop with the notebook's **Interrupt Kernel** button (⏹) — it's caught cleanly below.


### 9.1 Install real-time dependencies

In [11]:
%pip install -q sounddevice openwakeword numpy scipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 62.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 36.3 MB/s eta 0:00:00:00:0100:01


In [14]:
import openwakeword

print("Bundled wake word models:", list(openwakeword.models.keys()))
for name, info in openwakeword.models.items():
    print(f"  {name}: {info['model_path']}")

Bundled wake word models: ['alexa', 'hey_mycroft', 'hey_jarvis', 'timer', 'weather']
  alexa: /usr/local/lib/python3.13/dist-packages/openwakeword/resources/models/alexa_v0.1.onnx
  hey_mycroft: /usr/local/lib/python3.13/dist-packages/openwakeword/resources/models/hey_mycroft_v0.1.onnx
  hey_jarvis: /usr/local/lib/python3.13/dist-packages/openwakeword/resources/models/hey_jarvis_v0.1.onnx
  timer: /usr/local/lib/python3.13/dist-packages/openwakeword/resources/models/timer_v0.1.onnx
  weather: /usr/local/lib/python3.13/dist-packages/openwakeword/resources/models/weather_v0.1.onnx


### 9.2 Load the wake word model

First run downloads openWakeWord's pretrained model files. `"hey_jarvis"` is used here as the wake word — swap the model name for another bundled option (e.g. `"alexa"`, `"hey_mycroft"`) or train a custom one via the openWakeWord docs if you want a different phrase.


In [20]:
from openwakeword.model import Model as WakeWordModel
import openwakeword

print("Bundled/available wake word models:", list(openwakeword.models.keys()))

WAKE_WORD_NAME = "hey_jarvis"
model_path = openwakeword.models[WAKE_WORD_NAME]["model_path"]

wake_model = WakeWordModel(wakeword_model_paths=[model_path])
print("Loaded wake word models:", list(wake_model.models.keys()))

Bundled/available wake word models: ['alexa', 'hey_mycroft', 'hey_jarvis', 'timer', 'weather']


/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Loaded wake word models: ['hey_jarvis_v0.1']


### 9.3 Audio capture and voice-activity settings

Tune these against your own mic/room before relying on it. `SILENCE_TIMEOUT_S` is the most sensitive one — too short and it cuts off the end of commands, too long and it waits after the speaker is done.

In [21]:
import sounddevice as sd
import numpy as np
import queue, time, wave, tempfile, os as _os

SAMPLE_RATE = 16000              # required by both openWakeWord and SraVaani
BLOCK_MS = 80                    # audio chunk size fed to the wake word model
BLOCK_SIZE = int(SAMPLE_RATE * BLOCK_MS / 1000)

WAKE_THRESHOLD = 0.5              # openWakeWord score above which we treat it as triggered
SILENCE_RMS_THRESHOLD = 300       # int16 RMS below this counts as "quiet" -- calibrate to your mic/room
SILENCE_TIMEOUT_S = 1.2           # stop recording after this much continuous silence
MAX_COMMAND_S = 6.0               # hard cap so a stuck/noisy mic can't record forever
PRE_ROLL_BLOCKS = 3               # include a few blocks *before* the wake word trigger, to avoid clipping the command

def rms(int16_block: np.ndarray) -> float:
    return float(np.sqrt(np.mean(int16_block.astype(np.float32) ** 2)))

def list_input_devices():
    for i, d in enumerate(sd.query_devices()):
        if d["max_input_channels"] > 0:
            print(i, d["name"])

print("Available input devices:")
list_input_devices()
print("\nIf the wrong device gets picked, set sd.default.device = <index> before running the loop below.")


OSError: PortAudio library not found

### 9.4 The listen → record → classify loop

State machine with two states: `LISTENING` (scoring every block against the wake word) and `RECORDING` (buffering audio for the command that follows). When recording ends, the buffered clip is written to a temp WAV and passed straight into `pipeline()` from Section 5.


In [ ]:
def save_wav(path: str, audio_int16: np.ndarray, sample_rate: int = SAMPLE_RATE):
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)  # int16
        wf.setframerate(sample_rate)
        wf.writeframes(audio_int16.tobytes())

def run_wake_word_loop(on_result=None, max_iterations=None):
    """
    Blocking real-time loop. Interrupt the kernel to stop.

    on_result: optional callback(dict) called with each pipeline() result.
               Defaults to printing the result.
    max_iterations: optional int, stop after this many detected commands (useful for testing).
    """
    if on_result is None:
        on_result = lambda r: print(f"[{time.strftime('%H:%M:%S')}] '{r['transcript']}' -> {r['label']} (conf={r['confidence']})")

    audio_q = queue.Queue()

    def audio_callback(indata, frames, time_info, status):
        if status:
            print("Audio status:", status)
        audio_q.put(indata.copy())

    state = "LISTENING"
    pre_roll = []            # rolling buffer of recent blocks, for pre-roll on trigger
    command_buffer = []
    silence_start = None
    record_start = None
    n_results = 0

    print(f"Listening for wake word '{WAKE_WORD_NAME}' ... (interrupt the kernel to stop)")

    with sd.InputStream(samplerate=SAMPLE_RATE, channels=1, dtype="int16",
                         blocksize=BLOCK_SIZE, callback=audio_callback):
        try:
            while True:
                block = audio_q.get()
                block_1d = block[:, 0]

                if state == "LISTENING":
                    pre_roll.append(block_1d)
                    if len(pre_roll) > PRE_ROLL_BLOCKS:
                        pre_roll.pop(0)

                    scores = wake_model.predict(block_1d)
                    score = scores.get(WAKE_WORD_NAME, 0.0)

                    if score >= WAKE_THRESHOLD:
                        print(f"Wake word detected (score={score:.2f}). Recording command...")
                        state = "RECORDING"
                        command_buffer = list(pre_roll)   # seed with pre-roll so we don't clip the start
                        silence_start = None
                        record_start = time.time()

                elif state == "RECORDING":
                    command_buffer.append(block_1d)
                    level = rms(block_1d)
                    now = time.time()

                    if level < SILENCE_RMS_THRESHOLD:
                        if silence_start is None:
                            silence_start = now
                        elif now - silence_start >= SILENCE_TIMEOUT_S:
                            state = "DONE"
                    else:
                        silence_start = None

                    if now - record_start >= MAX_COMMAND_S:
                        print("Hit max command duration, cutting off.")
                        state = "DONE"

                    if state == "DONE":
                        audio = np.concatenate(command_buffer)
                        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                            tmp_path = tmp.name
                        save_wav(tmp_path, audio)

                        try:
                            result = pipeline(tmp_path, verbose=False)
                            on_result(result)
                        except Exception as e:
                            print("Pipeline error:", e)
                        finally:
                            _os.remove(tmp_path)

                        n_results += 1
                        command_buffer = []
                        pre_roll = []
                        state = "LISTENING"

                        if max_iterations is not None and n_results >= max_iterations:
                            print(f"Reached max_iterations={max_iterations}, stopping.")
                            break

        except KeyboardInterrupt:
            print("Stopped by user.")


### 9.5 Run it

Start with `max_iterations=3` to sanity-check detection and cutoff behavior before letting it run indefinitely.

In [ ]:
run_wake_word_loop(max_iterations=3)


In [ ]:
# Once tuned, run indefinitely -- interrupt the kernel (stop button) to end the session.
# run_wake_word_loop()


### 9.6 Tuning notes

- **False wake-ups or missed wake-ups:** adjust `WAKE_THRESHOLD`. Lower it if the wake word is missed, raise it if background noise or other speech falsely triggers it.
- **Commands getting cut off mid-word:** raise `SILENCE_TIMEOUT_S` slightly, or check `SILENCE_RMS_THRESHOLD` against your actual room noise floor — print `rms(block_1d)` for a few seconds of normal silence and normal speech to calibrate both values concretely rather than guessing.
- **Wrong microphone picked:** use `list_input_devices()` above, then set `sd.default.device = <index>` before calling `run_wake_word_loop()`.
- **Latency:** SraVaani + Laya inference happens synchronously after each command is captured, so there's a pause after the speaker finishes before you see a result. If that's a UX problem, the ASR call is the one to move to a background thread or a GPU.
- **This still needs the same evaluation discipline as Section 7.** A live mic introduces room echo, distance-from-mic variation, and overlapping speech that a clean test file won't have — re-run Section 7's evaluation using clips captured *through this same loop*, not just your original test files.
